# 06 - Processamento e validacao da atualizacao

Aplica gates a todas as coletas do recorte incremental, registra separadamente exclusoes de escopo e adiamentos exatos, regenera a fotografia `current`, valida os sete Parquets e arquiva os artefatos operacionais do ciclo.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
ACTIVE_CONFIG_PATH = DATA_ROOT / "operations" / "atualizacao" / "active.json"
REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_DIR = Path("/content/falando_nela")
REPO_REF = ""  # Opcional: branch, tag ou commit. Vazio usa o default remoto.

os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)
for name in ["raw", "checkpoints", "logs", "manifests", "processed", "operations/atualizacao"]:
    (DATA_ROOT / name).mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
print("DATA_ROOT:", DATA_ROOT)
print("Repositorio:", subprocess.run(["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip())

In [ ]:
EXPECTED_CYCLE_ID = "20260713"
if not ACTIVE_CONFIG_PATH.exists():
    raise FileNotFoundError(f"Controle ativo ausente: {ACTIVE_CONFIG_PATH}. Execute o caderno 00 primeiro.")
CONFIG = json.loads(ACTIVE_CONFIG_PATH.read_text(encoding="utf-8"))
assert CONFIG["schema_version"] == 1
assert CONFIG["cycle_id"] == EXPECTED_CYCLE_ID, CONFIG["cycle_id"]
assert CONFIG["window"] == {"data_inicio": "2026-05-01", "data_fim": "2026-07-13"}
assert CONFIG["data_inicio"] == CONFIG["window"]["data_inicio"]
assert CONFIG["data_fim"] == CONFIG["window"]["data_fim"]
assert Path(CONFIG["data_root"]) == DATA_ROOT
ALL_CONFIGURED_RUNS = {item["key"]: item for item in CONFIG["collection_runs"]}
PRESERVED_OUT_OF_SCOPE_RUNS = {
    item["key"]: item for item in CONFIG.get("preserved_out_of_scope_runs", [])
}
CAMARA_PLENARIO_HISTORICAL_SCOPE = {
    "key": "camara_plenario_historico",
    "source": "camara",
    "dataset": "plenario_discursos",
    "run_id": "prod-historico-camara-plenario",
    "data_inicio": "1946-01-01",
    "data_fim": "2026-05-28",
}
historical_candidate = (
    ALL_CONFIGURED_RUNS.get("camara_plenario_historico")
    or PRESERVED_OUT_OF_SCOPE_RUNS.get("camara_plenario_historico")
)
SCOPE_EXCLUDED_RUNS = {}
if historical_candidate is not None:
    for field, expected in CAMARA_PLENARIO_HISTORICAL_SCOPE.items():
        assert historical_candidate.get(field) == expected, {
            "field": field,
            "expected": expected,
            "actual": historical_candidate.get(field),
        }
    SCOPE_EXCLUDED_RUNS[historical_candidate["key"]] = historical_candidate
RUNS = {
    key: run
    for key, run in ALL_CONFIGURED_RUNS.items()
    if key not in SCOPE_EXCLUDED_RUNS
}
SCOPE_EXCLUSION_RESULTS = {
    key: {
        "run_id": run["run_id"],
        "status": "out_of_scope",
        "reason": (
            "O ciclo 20260713 e uma atualizacao incremental de no maximo tres meses; "
            "a recuperacao iniciada em 1946 nao e requisito desta atualizacao."
        ),
        "preserved_artifacts": ["raw", "checkpoint", "log", "autosave", "manifest"],
    }
    for key, run in SCOPE_EXCLUDED_RUNS.items()
}
print("Ciclo ativo:", CONFIG["cycle_id"], CONFIG["window"])
print("Coletas fora do escopo incremental:", SCOPE_EXCLUSION_RESULTS)

In [ ]:
from contextlib import contextmanager
from datetime import datetime, timezone

TERMINAL_STATUSES = {"completed"}
DEFERRED_COLLECTIONS_PATH = (
    DATA_ROOT
    / "operations"
    / "atualizacao"
    / "ciclos"
    / EXPECTED_CYCLE_ID
    / "deferred_collections.json"
)
DEFERRED_COLLECTION_POLICIES = {
    "senado_ccj_historico": {
        "run_id": "prod-historico-senado-ccj",
        "allowed_statuses": ["completed_with_errors"],
        "allowed_unresolved_partitions": ["2015-05"],
        "analysis_exclusion": "senado/ccj_notas",
    }
}

def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    content = path.read_text(encoding="utf-8")
    return json.loads(content) if content.strip() else None

def manifest_for(run):
    return DATA_ROOT / "manifests" / f"{run['run_id']}.json"

def checkpoint_for(run):
    return DATA_ROOT / "checkpoints" / run["source"] / f"{run['dataset']}.json"

def unresolved_partitions(run):
    checkpoint = read_json(checkpoint_for(run)) or {}
    current = (checkpoint.get("runs") or {}).get(run["run_id"], {}) or {}
    failed = set((current.get("failed_partitions") or {}).keys())
    completed = set((current.get("completed_partitions") or {}).keys())
    return sorted(failed - completed)

def _assert_manifest_contract(run):
    manifest = read_json(manifest_for(run))
    assert manifest is not None, f"Manifest final ausente: {manifest_for(run)}"
    assert manifest.get("run_id") == run["run_id"]
    assert manifest.get("mode") == "prod", (run["key"], manifest.get("mode"))
    assert manifest.get("sample") is False, (run["key"], manifest.get("sample"))
    assert manifest.get("data_inicio") == run["data_inicio"], (run["key"], manifest.get("data_inicio"))
    assert manifest.get("data_fim") == run["data_fim"], (run["key"], manifest.get("data_fim"))
    return manifest

def assert_collection_complete(run):
    manifest = _assert_manifest_contract(run)
    assert manifest.get("status") in TERMINAL_STATUSES, (run["key"], manifest.get("status"))
    unresolved = unresolved_partitions(run)
    assert not unresolved, f"Particoes falhas nao resolvidas em {run['key']}: {unresolved[:20]}"
    return manifest

def deferred_collection_for(run):
    payload = read_json(DEFERRED_COLLECTIONS_PATH)
    if not payload:
        return None
    assert payload.get("schema_version") == 1, DEFERRED_COLLECTIONS_PATH
    assert payload.get("cycle_id") == EXPECTED_CYCLE_ID, payload.get("cycle_id")
    for item in payload.get("items", []):
        if item.get("key") == run["key"]:
            policy = DEFERRED_COLLECTION_POLICIES.get(run["key"])
            assert policy is not None, f"Adiamento sem politica: {run['key']}"
            assert item.get("run_id") == run["run_id"] == policy["run_id"], item
            assert item.get("allowed_statuses") == policy["allowed_statuses"], item
            assert (
                item.get("allowed_unresolved_partitions")
                == policy["allowed_unresolved_partitions"]
            ), item
            assert item.get("analysis_exclusion") == policy["analysis_exclusion"], item
            return item
    return None

def collection_acceptance(run):
    try:
        manifest = assert_collection_complete(run)
        return {
            "manifest": manifest,
            "deferred": False,
            "status": manifest.get("status"),
            "unresolved": [],
        }
    except AssertionError as strict_error:
        deferral = deferred_collection_for(run)
        assert deferral is not None, strict_error
        manifest = _assert_manifest_contract(run)
        allowed_statuses = set(deferral.get("allowed_statuses") or [])
        allowed_unresolved = sorted(deferral.get("allowed_unresolved_partitions") or [])
        actual_unresolved = unresolved_partitions(run)
        assert deferral.get("analysis_excluded") is True, deferral
        assert str(deferral.get("reason") or "").strip(), deferral
        assert manifest.get("status") in allowed_statuses, (
            run["key"], manifest.get("status"), sorted(allowed_statuses)
        )
        assert actual_unresolved == allowed_unresolved, {
            "run": run["key"],
            "expected_unresolved": allowed_unresolved,
            "actual_unresolved": actual_unresolved,
        }
        return {
            "manifest": manifest,
            "deferred": True,
            "status": manifest.get("status"),
            "unresolved": actual_unresolved,
            "reason": deferral["reason"],
            "follow_up": deferral.get("follow_up"),
        }

def assert_collection_accepted(run):
    return collection_acceptance(run)["manifest"]

def show_run_state(run, tail_lines=5):
    final = read_json(manifest_for(run))
    autosave_path = DATA_ROOT / "manifests" / f"{run['run_id']}.autosave.json"
    autosave = read_json(autosave_path)
    log_path = DATA_ROOT / "logs" / f"{run['run_id']}.jsonl"
    tail = log_path.read_text(encoding="utf-8").splitlines()[-tail_lines:] if log_path.exists() else []
    print(run["key"], {
        "manifest": str(manifest_for(run)),
        "status": final.get("status") if final else None,
        "autosave_status": autosave.get("status") if autosave else None,
        "unresolved": unresolved_partitions(run),
        "log_tail": tail,
    })

def collector_command(run, *extra):
    return [
        sys.executable, "-u", "-m", run["module"],
        "--mode", "prod",
        "--output-dir", str(DATA_ROOT),
        "--data-inicio", run["data_inicio"],
        "--data-fim", run["data_fim"],
        "--run-id", run["run_id"],
        "--no-sample", "--resume", *extra,
    ]

def run_streamed(command, label):
    print(f"\n=== {label} ===", flush=True)
    print(" ".join(map(str, command)), flush=True)
    completed = subprocess.run(list(map(str, command)), check=False)
    returncode = completed.returncode
    print(f"=== retorno {returncode}: {label} ===", flush=True)
    return returncode

@contextmanager
def dataset_lock(run):
    lock_root = DATA_ROOT / "operations" / "atualizacao" / "locks"
    lock_root.mkdir(parents=True, exist_ok=True)
    lock_path = lock_root / f"{run['source']}__{run['dataset']}.json"
    payload = {
        "cycle_id": CONFIG["cycle_id"],
        "run_id": run["run_id"],
        "source": run["source"],
        "dataset": run["dataset"],
        "started_at": datetime.now(timezone.utc).isoformat(),
    }
    try:
        with lock_path.open("x", encoding="utf-8") as handle:
            json.dump(payload, handle, ensure_ascii=False, indent=2, sort_keys=True)
            handle.write("\n")
    except FileExistsError as exc:
        raise RuntimeError(f"Dataset ja bloqueado por outra sessao: {lock_path}\n{lock_path.read_text()}") from exc
    try:
        yield
    finally:
        if lock_path.exists() and read_json(lock_path) == payload:
            lock_path.unlink()

def run_collector(run, *extra):
    with dataset_lock(run):
        return run_streamed(collector_command(run, *extra), run["key"])

def require_explicit_confirmation(enabled, confirmation):
    if enabled:
        assert confirmation == EXPECTED_CYCLE_ID, "Digite o cycle_id na variavel CONFIRMAR_CICLO."

def assert_parlamentares_ready():
    run_id = CONFIG["processing_run_ids"]["parlamentares"]
    manifest_path = DATA_ROOT / "processed" / "manifests" / f"{run_id}-parlamentares.json"
    periodos_path = DATA_ROOT / "processed" / "parlamentares" / "v1" / "parquet" / "parlamentares_periodos.parquet"
    manifest = read_json(manifest_path)
    assert manifest and manifest.get("run_id") == run_id and manifest.get("dataset_version") == "v1", manifest_path
    assert periodos_path.exists(), periodos_path
    return manifest

## Gate bloqueante de coleta

In [ ]:
GATE_RESULTS = {}
for key, run in RUNS.items():
    try:
        acceptance = collection_acceptance(run)
        GATE_RESULTS[key] = {
            "ok": True,
            "status": acceptance["status"],
            "deferred": acceptance["deferred"],
            "unresolved": acceptance["unresolved"],
            "reason": acceptance.get("reason"),
            "manifest": str(manifest_for(run)),
        }
    except Exception as exc:
        GATE_RESULTS[key] = {
            "ok": False,
            "deferred": False,
            "error": str(exc),
            "manifest": str(manifest_for(run)),
        }

for key, result in GATE_RESULTS.items():
    print(key, result)
parlamentares_run = CONFIG["processing_run_ids"]["parlamentares"]
parlamentares_manifest = DATA_ROOT / "processed" / "manifests" / f"{parlamentares_run}-parlamentares.json"
parlamentares_periodos = DATA_ROOT / "processed" / "parlamentares" / "v1" / "parquet" / "parlamentares_periodos.parquet"
PARLAMENTARES_GATE_OK = parlamentares_manifest.exists() and parlamentares_periodos.exists()
COLLECTION_GATE_OK = all(item["ok"] for item in GATE_RESULTS.values()) and PARLAMENTARES_GATE_OK
STRICT_COLLECTION_GATE_OK = (
    COLLECTION_GATE_OK
    and not any(item.get("deferred") for item in GATE_RESULTS.values())
    and not SCOPE_EXCLUSION_RESULTS
)
DEFERRED_GATE_KEYS = sorted(
    key for key, item in GATE_RESULTS.items() if item.get("deferred")
)
SCOPE_EXCLUDED_GATE_KEYS = sorted(SCOPE_EXCLUSION_RESULTS)
print("PARLAMENTARES_GATE_OK=", PARLAMENTARES_GATE_OK)
print("COLLECTION_GATE_OK=", COLLECTION_GATE_OK)
print("STRICT_COLLECTION_GATE_OK=", STRICT_COLLECTION_GATE_OK)
print("DEFERRED_GATE_KEYS=", DEFERRED_GATE_KEYS)
print("SCOPE_EXCLUDED_GATE_KEYS=", SCOPE_EXCLUDED_GATE_KEYS)

## Auditoria JSONL bloqueante

A leitura pode ser longa no Drive, mas nao faz requisicoes externas nem altera o raw.

In [ ]:
RODAR_AUDITORIA_JSONL = False
CONFIRMAR_CICLO = ""
require_explicit_confirmation(RODAR_AUDITORIA_JSONL, CONFIRMAR_CICLO)
JSONL_GATE_OK = False
if RODAR_AUDITORIA_JSONL:
    assert COLLECTION_GATE_OK
    audit = {"cycle_id": EXPECTED_CYCLE_ID, "runs": {}, "invalid": []}
    for key, run in RUNS.items():
        sources = run.get("checkpoint_sources") or [run["source"]]
        paths = []
        for source in sources:
            paths.extend((DATA_ROOT / "raw" / source / run["dataset"]).rglob(f"{run['run_id']}.jsonl"))
        paths = sorted(set(paths))
        records = 0
        for path in paths:
            with path.open("r", encoding="utf-8") as handle:
                for line_number, line in enumerate(handle, start=1):
                    if not line.strip():
                        continue
                    try:
                        value = json.loads(line)
                        if not isinstance(value, dict):
                            raise ValueError("linha JSONL nao e objeto")
                        records += 1
                    except Exception as exc:
                        audit["invalid"].append({"path": str(path), "line": line_number, "error": str(exc)})
                        if len(audit["invalid"]) >= 100:
                            break
            if len(audit["invalid"]) >= 100:
                break
        audit["runs"][key] = {"files": len(paths), "records": records}
        print(key, audit["runs"][key])
        if not paths:
            audit["invalid"].append({"run": key, "error": "nenhum JSONL encontrado"})
    JSONL_GATE_OK = not audit["invalid"]
    audit["ok"] = JSONL_GATE_OK
    audit_path = DATA_ROOT / "operations" / "atualizacao" / "ciclos" / EXPECTED_CYCLE_ID / "audits" / "raw_jsonl.json"
    audit_path.parent.mkdir(parents=True, exist_ok=True)
    audit_path.write_text(json.dumps(audit, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    assert JSONL_GATE_OK, audit["invalid"][:20]
    print("JSONL_GATE_OK=True", audit_path)
else:
    print("Auditoria protegida: RODAR_AUDITORIA_JSONL=False")

In [ ]:
RODAR_PROCESSAMENTO = False
CONFIRMAR_CICLO = ""
require_explicit_confirmation(RODAR_PROCESSAMENTO, CONFIRMAR_CICLO)
if RODAR_PROCESSAMENTO:
    assert COLLECTION_GATE_OK, "Processamento bloqueado pelos gates de coleta."
    assert JSONL_GATE_OK, "Processamento bloqueado ate a auditoria JSONL desta sessao passar."
    process_runs = CONFIG["processing_run_ids"]
    commands = [
        ([sys.executable, "-u", "-m", "processamento.normalizacao", "--mode", "prod", "--data-root", str(DATA_ROOT), "--run-id", process_runs["textos"], "--overwrite"], "normalizar textos current"),
        ([sys.executable, "-u", "-m", "processamento.apartes_parlamentares", "--mode", "prod", "--data-root", str(DATA_ROOT), "--run-id", process_runs["apartes"], "--overwrite"], "processar apartes current"),
        ([sys.executable, "-u", "-m", "processamento.parquet", "--profile", "colab", "--data-root", str(DATA_ROOT), "--run-id", process_runs["parquet"], "--overwrite"], "gerar sete Parquets"),
        ([sys.executable, "-u", "-m", "processamento.parlamentares_join_audit", "--profile", "colab", "--data-root", str(DATA_ROOT), "--run-id", process_runs["join_audit"], "--overwrite"], "auditar joins"),
        ([sys.executable, "-u", "-m", "processamento.samples", "--profile", "colab", "--data-root", str(DATA_ROOT), "--run-id", process_runs["samples"], "--include-parquet", "--overwrite"], "gerar ZIPs de amostra"),
    ]
    for command, label in commands:
        assert run_streamed(command, label) == 0, label
else:
    print("Processamento protegido: RODAR_PROCESSAMENTO=False")

## Validacao dos sete Parquets e dos derivados

In [ ]:
import pyarrow.compute as pc
import pyarrow.parquet as pq

parquet_root = DATA_ROOT / "processed" / "textos_parlamentares" / "v1" / "parquet"
actual_names = sorted(path.name for path in parquet_root.glob("*.parquet"))
expected_names = sorted(CONFIG["expected_text_parquets"])
assert actual_names == expected_names, {"expected": expected_names, "actual": actual_names}

total_rows = 0
seen_ids = set()
for name in expected_names:
    path = parquet_root / name
    expected_source, expected_dataset = name.removesuffix(".parquet").split("__", maxsplit=1)
    table = pq.read_table(path, columns=["texto_id", "source", "dataset", "dataset_version", "texto"])
    ids = table.column("texto_id").to_pylist()
    versions = set(table.column("dataset_version").to_pylist())
    texts = table.column("texto").to_pylist()
    assert set(table.column("source").to_pylist()) == {expected_source}
    assert set(table.column("dataset").to_pylist()) == {expected_dataset}
    assert len(ids) == len(set(ids)), f"texto_id duplicado dentro de {name}"
    overlap = seen_ids.intersection(ids)
    assert not overlap, f"texto_id repetido entre bases: {next(iter(overlap))}"
    seen_ids.update(ids)
    assert versions == {"v1"}, (name, versions)
    assert all(isinstance(value, str) and value.strip() for value in texts), f"texto vazio em {name}"
    total_rows += table.num_rows
    print(name, table.num_rows)

normal_manifest = read_json(DATA_ROOT / "processed" / "manifests" / f"{CONFIG['processing_run_ids']['textos']}.json")
assert normal_manifest
assert normal_manifest.get("run_id") == CONFIG["processing_run_ids"]["textos"]
assert normal_manifest.get("dataset_version") == "v1"
assert total_rows == normal_manifest.get("output_records"), (total_rows, normal_manifest)
expected_raw_run_ids = {
    run["run_id"]
    for run in RUNS.values()
    if f"{run['source']}/{run['dataset']}" in CONFIG["expected_text_datasets"]
}
observed_raw_run_ids = set(normal_manifest.get("raw_run_ids", []))
assert expected_raw_run_ids <= observed_raw_run_ids, sorted(expected_raw_run_ids - observed_raw_run_ids)

initial_inventory_path = DATA_ROOT / "operations" / "atualizacao" / "ciclos" / EXPECTED_CYCLE_ID / "audits" / "initial_inventory.json"
initial_inventory = read_json(initial_inventory_path) or {}
previous_total_rows = int(initial_inventory.get("previous_text_rows") or 0)
JUSTIFICATIVA_REDUCAO = ""  # Preencha somente se uma reducao tiver sido investigada e aceita.
if total_rows < previous_total_rows:
    assert JUSTIFICATIVA_REDUCAO.strip(), (previous_total_rows, total_rows)
ROW_COMPARISON = {
    "previous_text_rows": previous_total_rows,
    "current_text_rows": total_rows,
    "delta": total_rows - previous_total_rows,
    "reduction_justification": JUSTIFICATIVA_REDUCAO or None,
}

proc_runs = CONFIG["processing_run_ids"]
parlamentares_path = DATA_ROOT / "processed" / "manifests" / f"{proc_runs['parlamentares']}-parlamentares.json"
apartes_manifest_path = DATA_ROOT / "processed" / "manifests" / f"{proc_runs['apartes']}-apartes-parlamentares.json"
apartes_path = DATA_ROOT / "processed" / "apartes_parlamentares" / "v1" / "parquet" / "apartes_parlamentares.parquet"
join_path = DATA_ROOT / "processed" / "audits" / "parlamentares" / proc_runs["join_audit"] / "manifest.json"
samples_path = DATA_ROOT / "processed" / "downloads" / proc_runs["samples"] / "manifest.json"
required_paths = [parlamentares_path, apartes_manifest_path, apartes_path, join_path, samples_path]
for path in required_paths:
    assert path.exists(), path

parlamentares_manifest = read_json(parlamentares_path)
assert parlamentares_manifest.get("run_id") == proc_runs["parlamentares"]
assert parlamentares_manifest.get("dataset_version") == "v1"
assert all(Path(path).exists() for path in parlamentares_manifest.get("output_files", {}).values())
assert all(Path(path).exists() for path in parlamentares_manifest.get("parquet_files", {}).values())
apartes_manifest = read_json(apartes_manifest_path)
apartes_table = pq.read_table(apartes_path, columns=["source"])
assert apartes_manifest.get("run_id") == proc_runs["apartes"]
assert apartes_manifest.get("dataset_version") == "v1"
assert all(Path(path).exists() for path in apartes_manifest.get("output_files", {}).values())
assert all(Path(path).exists() for path in apartes_manifest.get("parquet_files", {}).values())
assert apartes_table.num_rows == apartes_manifest.get("output_records")
assert set(apartes_table.column("source").to_pylist()) == set(CONFIG["expected_apartes_sources"])
join_manifest = read_json(join_path)
assert join_manifest.get("run_id") == proc_runs["join_audit"]
assert join_manifest.get("textos_lidos") == total_rows
samples_manifest = read_json(samples_path)
assert samples_manifest.get("run_id") == proc_runs["samples"]
assert samples_manifest.get("dataset_version") == "v1"
sample_groups = samples_manifest.get("output_record_counts", {})
for dataset in CONFIG["expected_text_datasets"]:
    prefix = dataset.replace("/", "__") + "__"
    assert any(key.startswith(prefix) for key in sample_groups), f"Amostra ausente: {dataset}"
assert all(Path(path).exists() for path in samples_manifest.get("output_files", []))
print("Validacao estrutural aprovada:", total_rows, "textos unicos", ROW_COMPARISON)

## Arquivamento do ciclo

Copia somente configuracao, manifests e auditorias; raw e fotografias `current` nao sao duplicados.

In [ ]:
import shutil

ARQUIVAR_CICLO = False
CONFIRMAR_CICLO = ""
require_explicit_confirmation(ARQUIVAR_CICLO, CONFIRMAR_CICLO)
if ARQUIVAR_CICLO:
    cycle_dir = DATA_ROOT / "operations" / "atualizacao" / "ciclos" / EXPECTED_CYCLE_ID
    archive_manifests = cycle_dir / "manifests"
    archive_audits = cycle_dir / "audits"
    archive_manifests.mkdir(parents=True, exist_ok=True)
    archive_audits.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ACTIVE_CONFIG_PATH, cycle_dir / "config.json")
    manifest_paths = [manifest_for(run) for run in RUNS.values()]
    manifest_paths.extend([
        Path(normal_manifest["manifest_path"]),
        parlamentares_path,
        apartes_manifest_path,
        DATA_ROOT / "processed" / "manifests" / f"{CONFIG['processing_run_ids']['parquet']}-parquet.json",
    ])
    for path in manifest_paths:
        if path.exists():
            shutil.copy2(path, archive_manifests / path.name)
    for archive_name, path in {
        "join-audit-manifest.json": join_path,
        "samples-manifest.json": samples_path,
    }.items():
        shutil.copy2(path, archive_manifests / archive_name)
    cycle_audit_roots = [
        DATA_ROOT / "processed" / "audits" / "parlamentares" / CONFIG["processing_run_ids"]["join_audit"],
        DATA_ROOT / "processed" / "audits" / "apartes_parlamentares" / CONFIG["processing_run_ids"]["apartes"],
    ]
    for audit_root in cycle_audit_roots:
        for source in audit_root.rglob("*") if audit_root.exists() else []:
            if source.is_file():
                target = archive_audits / audit_root.name / source.relative_to(audit_root)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, target)
    GRADIO_CHECKLIST = {
        "new_window_records_found": False,
        "compact_table_omits_text": False,
        "full_text_opened_by_texto_id": False,
    }
    summary = {
        "cycle_id": EXPECTED_CYCLE_ID,
        "config": CONFIG,
        "window": CONFIG["window"],
        "collection_gate": GATE_RESULTS,
        "collection_gate_ok": COLLECTION_GATE_OK,
        "strict_collection_gate_ok": STRICT_COLLECTION_GATE_OK,
        "deferred_collection_keys": DEFERRED_GATE_KEYS,
        "deferred_collections": read_json(DEFERRED_COLLECTIONS_PATH),
        "scope_excluded_collection_keys": SCOPE_EXCLUDED_GATE_KEYS,
        "scope_excluded_collections": SCOPE_EXCLUSION_RESULTS,
        "jsonl_gate_ok": JSONL_GATE_OK,
        "expected_text_parquets": CONFIG["expected_text_parquets"],
        "row_comparison": ROW_COMPARISON,
        "processed_manifests": {
            "textos": normal_manifest.get("manifest_path"),
            "parlamentares": str(parlamentares_path),
            "apartes": str(apartes_manifest_path),
            "join_audit": str(join_path),
            "samples": str(samples_path),
        },
        "manual_gradio_inspection": GRADIO_CHECKLIST,
    }
    (cycle_dir / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print("Ciclo arquivado em", cycle_dir)

## Inspecao final no Gradio

Ative somente depois da validacao. Busque datas da nova janela, confira que a tabela compacta nao contem `texto` e abra o texto integral por `texto_id`.

In [ ]:
ABRIR_GRADIO = False
if ABRIR_GRADIO:
    assert COLLECTION_GATE_OK
    from processamento.visualizador_parquets import build_gradio_app
    app = build_gradio_app(parquet_root)
    app.launch(share=True)